# 00 — First principles of feedback intelligence

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/09_feedback_intelligence/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and NLP familiarity (sentiment, embeddings)
- Understanding of text processing and classification

**What you will learn**:
- Why feedback intelligence goes far beyond simple positive / negative sentiment
- How aspect-level opinion mining decomposes reviews into actionable tuples
- Why simple sentiment classifiers destroy critical business insights
- How temporal trend detection surfaces emerging issues before they escalate
- The feedback-to-action loop that converts signals into prioritised work items
- The $12 B+ CXM market opportunity (Medallia, Qualtrics) and the mid-market gap

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0" "matplotlib>=3.7.0" "pandas>=2.0.0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import defaultdict, Counter
import re, textwrap, json, statistics

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — What is feedback intelligence?

Traditional analytics asks *"Are customers happy?"*. Feedback intelligence asks
*"What specifically are customers happy or unhappy about, how intensely, and is
it getting better or worse?"*

A single review like "Great product but terrible customer support — took 3 days
to get a response" contains **multiple signals**: positive product sentiment,
negative support sentiment, and a measurable SLA complaint. Feedback intelligence
decomposes every piece of feedback into **(aspect, opinion, sentiment, intensity)**
tuples that can be aggregated, trended, and acted upon.

Let's build the foundational schema and decompose an example review.

In [ ]:
@dataclass
class AspectOpinion:
    """A single aspect-level opinion extracted from feedback."""
    aspect: str                    # e.g., "customer support"
    opinion: str                   # e.g., "took 3 days to respond"
    sentiment: str                 # positive | negative | neutral
    intensity: float               # 0.0 – 1.0 (mild → extreme)
    evidence: str = ""             # verbatim quote

@dataclass
class FeedbackItem:
    """Normalised feedback from any source."""
    id: str
    text: str
    source: str                    # review | ticket | survey | social
    timestamp: str = ""
    metadata: Dict = field(default_factory=dict)

def decompose_review(text: str) -> List[AspectOpinion]:
    """Rule-based aspect extractor — illustrative, not production."""
    aspects: List[AspectOpinion] = []
    # Simple clause splitting + keyword heuristics
    clauses = re.split(r"[.!;]|\bbut\b|\bhowever\b", text, flags=re.IGNORECASE)
    positive_kw = {"great", "love", "excellent", "fast", "amazing", "good", "easy"}
    negative_kw = {"terrible", "slow", "bad", "awful", "poor", "broken", "frustrating"}

    for clause in clauses:
        clause = clause.strip()
        if not clause or len(clause) < 5:
            continue
        words = set(clause.lower().split())
        pos_hits = words & positive_kw
        neg_hits = words & negative_kw
        if pos_hits:
            sent, intensity = "positive", min(0.5 + 0.15 * len(pos_hits), 1.0)
        elif neg_hits:
            sent, intensity = "negative", min(0.5 + 0.15 * len(neg_hits), 1.0)
        else:
            sent, intensity = "neutral", 0.3
        aspects.append(AspectOpinion(
            aspect=clause[:40].strip(), opinion=clause, sentiment=sent, intensity=intensity
        ))
    return aspects

# --- Demo ---
review = "Great product but terrible customer support — took 3 days to get a response. Love the UI design."
aspects = decompose_review(review)
print(f"Review: {review}\n")
for a in aspects:
    print(f"  [{a.sentiment.upper():>8} {a.intensity:.1f}] {a.aspect}")
print(f"\n✓ Extracted {len(aspects)} aspect-level opinions from 1 review")

## Section 2 — Why simple sentiment fails

Most analytics tools reduce a review to a single sentiment score. This is
equivalent to averaging temperatures across a hospital — the average might be
"normal" while some patients have a fever and others are hypothermic.

Let's build a simple sentiment classifier and demonstrate the information loss
on a review that contains **both positive and negative** signals.

In [ ]:
def simple_sentiment(text: str) -> Dict[str, float]:
    """Bag-of-words sentiment scorer. Returns score in [-1, 1]."""
    positive_words = {
        "great": 0.8, "love": 0.9, "excellent": 0.9, "amazing": 0.95,
        "good": 0.6, "fast": 0.5, "easy": 0.5, "wonderful": 0.9, "best": 0.85,
    }
    negative_words = {
        "terrible": -0.9, "awful": -0.95, "bad": -0.7, "slow": -0.5,
        "poor": -0.6, "broken": -0.8, "frustrating": -0.7, "worst": -0.95,
    }
    words = text.lower().split()
    scores = []
    for w in words:
        w_clean = re.sub(r"[^a-z]", "", w)
        if w_clean in positive_words:
            scores.append(positive_words[w_clean])
        elif w_clean in negative_words:
            scores.append(negative_words[w_clean])
    if not scores:
        return {"score": 0.0, "label": "neutral", "detail": "no signal"}
    avg = sum(scores) / len(scores)
    label = "positive" if avg > 0.1 else ("negative" if avg < -0.1 else "neutral")
    return {"score": round(avg, 3), "label": label, "detail": f"{len(scores)} signal words"}

# --- Demo: information loss ---
reviews = [
    "Great product but terrible customer support",
    "Love the UI, love the speed, love everything!",
    "Awful experience from start to finish",
    "Product is good but billing is broken and support is slow",
]

print("Simple sentiment vs. reality:\n")
for r in reviews:
    result = simple_sentiment(r)
    aspects = decompose_review(r)
    neg_count = sum(1 for a in aspects if a.sentiment == "negative")
    pos_count = sum(1 for a in aspects if a.sentiment == "positive")
    print(f"  Score: {result['score']:+.2f} ({result['label']:>8})  |  "
          f"Aspects: {pos_count}+ / {neg_count}−  |  "{r[:50]}..."")

print("\n⚠ Notice: mixed reviews get neutral scores, hiding actionable negatives")

## Section 3 — Aspect-level opinion mining

The core extraction task is: given raw text, produce a list of
**(aspect, opinion, sentiment, intensity)** tuples. This is far richer than
document-level sentiment.

First we'll show the limitations of a regex/heuristic approach, then
demonstrate how an LLM-based extractor (prompt engineering) produces
structured output.

In [ ]:
# --- Heuristic extractor (limited) ---

ASPECT_KEYWORDS: Dict[str, List[str]] = {
    "product_quality": ["product", "quality", "build", "material", "durable"],
    "customer_support": ["support", "service", "help", "agent", "response"],
    "pricing": ["price", "cost", "expensive", "cheap", "value", "billing"],
    "ui_ux": ["ui", "ux", "interface", "design", "usability", "navigation"],
    "performance": ["speed", "fast", "slow", "lag", "performance", "crash"],
    "onboarding": ["setup", "onboarding", "install", "getting started"],
}

def heuristic_extractor(text: str) -> List[Dict]:
    """Map clauses to predefined aspects via keyword overlap."""
    results = []
    clauses = re.split(r"[.!;]|\bbut\b|\bhowever\b", text, flags=re.IGNORECASE)
    positive_kw = {"great", "love", "excellent", "fast", "amazing", "good", "easy"}
    negative_kw = {"terrible", "slow", "bad", "awful", "poor", "broken", "frustrating"}

    for clause in clauses:
        words = set(clause.lower().split())
        if len(words) < 2:
            continue
        # Find best matching aspect
        best_aspect, best_score = "general", 0
        for aspect, kws in ASPECT_KEYWORDS.items():
            overlap = len(words & set(kws))
            if overlap > best_score:
                best_aspect, best_score = aspect, overlap
        # Sentiment
        pos = words & positive_kw
        neg = words & negative_kw
        sent = "positive" if pos and not neg else ("negative" if neg else "neutral")
        results.append({"aspect": best_aspect, "sentiment": sent, "text": clause.strip()})
    return results

# --- Demo: heuristic limits ---
test_reviews = [
    "The dashboard is beautiful but the API keeps crashing under load.",
    "Love your product! But I've been waiting 5 days for a refund.",
    "Setup was a breeze. Pricing is way too high for small teams though.",
]

print("Heuristic extractor results:\n")
for r in test_reviews:
    extractions = heuristic_extractor(r)
    print(f"  Review: "{r}"")
    for e in extractions:
        print(f"    → [{e['sentiment']:>8}] {e['aspect']}: {e['text'][:60]}")
    print()

# --- LLM-based extractor (prompt only, no API call) ---
EXTRACTION_PROMPT = '''You are an aspect-level opinion mining system.
Given a customer feedback text, extract ALL aspect-opinion pairs.

Return JSON array:
[
  {{
    "aspect": "<category>",
    "opinion": "<what they said>",
    "sentiment": "positive|negative|neutral",
    "intensity": <0.0-1.0>,
    "evidence": "<verbatim quote>"
  }}
]

Categories: product_quality, customer_support, pricing, ui_ux,
performance, onboarding, documentation, reliability, general

Feedback: "{text}"
'''

print("--- LLM Extraction Prompt Template ---")
print(EXTRACTION_PROMPT.format(text="<customer feedback goes here>"))
print("✓ Heuristic extractor demonstrated; LLM prompt template ready")

## Section 4 — Trend detection as anomaly detection

Customer feedback is inherently **temporal**. A sudden spike in "login failure"
mentions is more urgent than a steady trickle of "UI colour" preferences.

Treating aspect mentions as a time series, we can apply **moving average
smoothing** and **z-score anomaly detection** to surface emerging issues before
they become crises.

In [ ]:
# --- Simulate 30 days of aspect mention counts ---
days = 30
aspects_list = ["product_quality", "customer_support", "pricing", "performance", "login"]

# Normal baseline + injected anomaly: login bugs spike from day 15
np.random.seed(42)
data: Dict[str, np.ndarray] = {}
for asp in aspects_list:
    base = np.random.poisson(lam=5, size=days).astype(float)
    if asp == "login":
        # Inject emerging issue starting day 15
        base[15:] += np.linspace(0, 15, days - 15) + np.random.normal(0, 1, days - 15)
    if asp == "pricing":
        # Seasonal: spike at month end (days 28–30)
        base[27:] += np.array([8, 12, 10])[:days - 27]
    data[asp] = np.clip(base, 0, None)

df_trends = pd.DataFrame(data, index=pd.date_range("2025-01-01", periods=days, freq="D"))
df_trends.index.name = "date"

def detect_anomalies(series: pd.Series, window: int = 7, z_thresh: float = 2.0) -> pd.Series:
    """Flag anomalies where z-score exceeds threshold vs rolling window."""
    rolling_mean = series.rolling(window=window, min_periods=3).mean()
    rolling_std = series.rolling(window=window, min_periods=3).std().replace(0, 1)
    z_scores = (series - rolling_mean) / rolling_std
    return z_scores > z_thresh

# --- Detect and visualise ---
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Top: all aspect trends
for asp in aspects_list:
    axes[0].plot(df_trends.index, df_trends[asp], label=asp, alpha=0.8)
axes[0].set_title("Aspect mention counts over 30 days")
axes[0].set_ylabel("Mentions / day")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Bottom: login anomaly detection
login = df_trends["login"]
rolling_mean = login.rolling(window=7, min_periods=3).mean()
anomalies = detect_anomalies(login)

axes[1].plot(df_trends.index, login, label="login mentions", color="tab:red")
axes[1].plot(df_trends.index, rolling_mean, label="7-day MA", color="black", linestyle="--")
axes[1].scatter(df_trends.index[anomalies], login[anomalies],
                color="red", s=80, zorder=5, label="anomaly", marker="^")
axes[1].set_title("Login issue — anomaly detection (z-score > 2.0)")
axes[1].set_ylabel("Mentions / day")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report anomalies
print("\nDetected anomalies:")
for asp in aspects_list:
    flags = detect_anomalies(df_trends[asp])
    n = flags.sum()
    if n > 0:
        first_day = df_trends.index[flags].min().strftime("%Y-%m-%d")
        print(f"  {asp}: {n} anomalous days, first on {first_day}")
print("\n✓ Trend detection with moving average and z-score anomaly flagging")

## Section 5 — The feedback-to-action loop

Insights without action are waste. The final step in feedback intelligence is
**priority scoring** that ranks issues by business impact so teams know what to
fix first.

**Priority = Impact × Trend × Actionability**

- **Impact** = volume × average severity (high-severity issues affecting many
  customers rank highest)
- **Trend** = is it getting worse? A growing trend multiplies urgency.
- **Actionability** = can the team actually fix this? Vague "it sucks" is less
  actionable than "login timeout after 30 s."

Let's build a priority ranker and apply it to our simulated data.

In [ ]:
@dataclass
class IssueSummary:
    """Aggregated issue for priority ranking."""
    aspect: str
    mention_count: int
    avg_severity: float        # 0–1
    trend_slope: float         # positive = worsening
    actionability: float       # 0–1 (heuristic or LLM-scored)

def priority_score(issue: IssueSummary) -> float:
    """Compute composite priority score."""
    impact = issue.mention_count * issue.avg_severity
    trend_multiplier = 1.0 + max(0, issue.trend_slope) * 2.0   # worsening doubles weight
    action_weight = 0.3 + 0.7 * issue.actionability            # floor at 0.3
    return round(impact * trend_multiplier * action_weight, 2)

# Build issues from our simulated data
issues = [
    IssueSummary("login", int(df_trends["login"].sum()), 0.85,
                 trend_slope=0.6, actionability=0.9),
    IssueSummary("customer_support", int(df_trends["customer_support"].sum()), 0.65,
                 trend_slope=0.05, actionability=0.7),
    IssueSummary("pricing", int(df_trends["pricing"].sum()), 0.55,
                 trend_slope=0.15, actionability=0.5),
    IssueSummary("product_quality", int(df_trends["product_quality"].sum()), 0.4,
                 trend_slope=-0.05, actionability=0.8),
    IssueSummary("performance", int(df_trends["performance"].sum()), 0.7,
                 trend_slope=0.02, actionability=0.85),
]

ranked = sorted(issues, key=priority_score, reverse=True)
print("Priority ranking (highest first):\n")
print(f"  {'Aspect':<20} {'Mentions':>8} {'Severity':>8} {'Trend':>8} {'Action':>8} {'PRIORITY':>10}")
print("  " + "─" * 74)
for iss in ranked:
    score = priority_score(iss)
    trend_arrow = "↑" if iss.trend_slope > 0.1 else ("→" if iss.trend_slope > -0.1 else "↓")
    print(f"  {iss.aspect:<20} {iss.mention_count:>8} {iss.avg_severity:>8.2f} "
          f"{trend_arrow:>8} {iss.actionability:>8.2f} {score:>10.1f}")

print(f"\n✓ Top priority: {ranked[0].aspect} (score {priority_score(ranked[0]):.1f})")
print("  The login bug is high-volume, high-severity, worsening, and fixable.")

## Section 6 — Market analysis

Customer Experience Management (CXM) is a **$12 B+** market defined by massive
signal volume and slow, manual triage:

| Signal | Scale | Challenge |
|---|---|---|
| Product reviews (App Store, G2, Capterra) | Millions per quarter | Multi-aspect, informal language |
| Support tickets (Zendesk, Freshdesk) | Thousands per day for mid-market SaaS | Routing, deduplication, prioritisation |
| Survey responses (NPS, CSAT) | High volume, low completion rate | Structured + free-text hybrid |
| Social mentions (Twitter/X, Reddit) | Unbounded | Noisy, sarcasm, context-dependent |

**Medallia** was acquired for **$6.4 B** (2021) and **Qualtrics** for **$12.5 B**
(2023) — demonstrating massive enterprise appetite for customer feedback
intelligence.

**Why mid-market is underserved**: Medallia and Qualtrics are enterprise-priced
($100K+ / year). Mid-market companies (1K–50K reviews/month) rely on
spreadsheets or basic dashboards. LLM-powered aspect extraction and trend
detection can deliver 80 % of the insight at 5 % of the cost.

In [ ]:
# --- Market sizing model ---

market_segments = {
    "Enterprise (50K+ reviews/mo)": {"companies": 5_000, "arpu": 120_000, "penetration": 0.35},
    "Mid-Market (5K-50K reviews/mo)": {"companies": 50_000, "arpu": 12_000, "penetration": 0.08},
    "SMB (500-5K reviews/mo)": {"companies": 500_000, "arpu": 1_200, "penetration": 0.02},
}

print("CXM Feedback Intelligence — Market Sizing\n")
print(f"  {'Segment':<35} {'Companies':>10} {'ARPU':>10} {'Penetration':>12} {'TAM ($M)':>10}")
print("  " + "─" * 80)
total_tam = 0
total_addressable = 0
for seg, d in market_segments.items():
    tam = d["companies"] * d["arpu"] / 1_000_000
    addressable = tam * d["penetration"]
    total_tam += tam
    total_addressable += addressable
    print(f"  {seg:<35} {d['companies']:>10,} ${d['arpu']:>9,} {d['penetration']:>11.0%} ${tam:>9,.0f}")

print("  " + "─" * 80)
print(f"  {'Total TAM':<35} {'':<10} {'':<10} {'':<12} ${total_tam:>9,.0f}")
print(f"  {'Current addressable (penetrated)':<35} {'':<10} {'':<10} {'':<12} ${total_addressable:>9,.0f}")

# Cost comparison
print("\n\nCost per 1,000 reviews analysed:")
methods = [
    ("Manual analyst (8 reviews/hr, $45/hr)", 1000 / 8 * 45),
    ("Legacy NLP pipeline (rules + ML)", 15.0),
    ("LLM-based extraction (GPT-4o-mini)", 2.50),
]
for method, cost in methods:
    print(f"  {method:<50} ${cost:>8,.2f}")
print("\n✓ LLM approach is 2,000× cheaper than manual analysis")

In [ ]:
# --- Quick self-check: verify key concepts ---
concepts = {
    "aspect_extraction": len(aspects) > 0,
    "trend_detection": df_trends.shape[0] == 30,
    "priority_ranking": priority_score(ranked[0]) > priority_score(ranked[-1]),
    "market_sizing": total_tam > 1000,
}
for concept, passed in concepts.items():
    status = "✓" if passed else "✗"
    print(f"  {status} {concept}")
print(f"\n✓ {sum(concepts.values())}/{len(concepts)} concept checks passed")

## Exercises

1. **Multi-aspect extraction** — Take 5 real product reviews (from Amazon,
   G2, or App Store) and manually extract all aspect-opinion tuples. Compare
   your human labels with the heuristic extractor's output. Compute precision
   and recall for aspect detection.

2. **Anomaly detection tuning** — Experiment with the z-score threshold (1.5,
   2.0, 2.5, 3.0) and window size (3, 5, 7, 14 days). Plot a precision-recall
   curve: lower thresholds catch more issues but generate false alarms.

3. **Priority formula design** — Design an alternative priority formula that
   incorporates customer segment (enterprise vs SMB) and revenue impact. Test
   it on the 5 issues and compare rankings with the original formula.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Feedback intelligence decomposes text into (aspect, opinion, sentiment, intensity) tuples |
| 2 | Simple sentiment averaging destroys critical mixed-signal information |
| 3 | Aspect-level extraction enables per-feature trend tracking and targeted action |
| 4 | Temporal anomaly detection (z-score over moving average) surfaces emerging issues early |
| 5 | Priority = Impact × Trend × Actionability converts signals into a ranked backlog |
| 6 | The $12 B+ CXM market is ripe for LLM disruption, especially in the underserved mid-market |

## What's Next

In **01 — Architecture**, we design the full feedback intelligence pipeline:
multi-source ingestion, LLM-based aspect extraction, trend detection,
alert engine, and executive summary generation.

---

## References

1. Thoma Bravo, *Medallia Acquisition — $6.4 B*, 2021 — <https://www.medallia.com/>
2. Silver Lake & CPP, *Qualtrics Acquisition — $12.5 B*, 2023 — <https://www.qualtrics.com/>
3. Pontiki, M. et al., *SemEval-2016 Task 5: Aspect-Based Sentiment Analysis*, 2016 — <https://aclanthology.org/S16-1002/>
4. Gartner, *Voice of the Customer Market Guide*, 2024 — <https://www.gartner.com/>